# Discover zoonotic bacterial species

This notebook searches PubMed for zoonosis-related articles and asks an LLM to extract bacterial species explicitly associated with zoonosis. It uses two levels: a title-only pass for candidate discovery, followed by an abstract pass for evidence and species names requiring context.

PubMed records, LLM decisions, and the final one-row-per-species dataframe are persisted locally. The last cell reconstructs the result from files, so it remains usable after manually interrupting either LLM stage.

In [1]:
%load_ext autoreload
%autoreload 2

import json
import os
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'assets').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'assets').exists():
    raise FileNotFoundError('Could not locate the repository assets directory.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from graphicalizer import (
    DEFAULT_ZOONOSIS_QUERY,
    OpenAIChatCompleter,
    PubMedClient,
    build_bacterial_species_summary,
    collect_zoonosis_articles,
    run_species_extraction_stage,
)
from graphicalizer.notebook_config import debug_pathogens, limit_debug_abstracts


## Configure PubMed, the LLM, and checkpoints

In [2]:
from graphicalizer.notebook_config import (
    configured_env,
    debug_pathogens,
    load_notebook_config,
    resolve_config_path,
)

CONFIG = load_notebook_config(PROJECT_ROOT)
COMMON = CONFIG['common']
SETTINGS = CONFIG['notebook_01_zoonotic_bacterial_species']
DEBUG_MODE = COMMON['debug_mode']
DEBUG_PATHOGENS = debug_pathogens(COMMON)
DEBUG_ABSTRACTS_PER_PATHOGEN = COMMON['debug_abstracts_per_pathogen']

PUBMED_EMAIL = COMMON['pubmed_email']
PUBMED_API_KEY = configured_env(COMMON, 'pubmed_api_key_env')
PUBMED_MAX_RESULTS = (
    DEBUG_ABSTRACTS_PER_PATHOGEN * len(DEBUG_PATHOGENS)
    if DEBUG_MODE
    else SETTINGS['pubmed_max_results']
)
PUBMED_SEARCH_PAGE_SIZE = COMMON['pubmed_search_page_size']
PUBMED_FETCH_BATCH_SIZE = COMMON['pubmed_fetch_batch_size']
QUERY = SETTINGS['query']

OUTPUT_DIR = resolve_config_path(
    PROJECT_ROOT,
    Path(COMMON['output_root'])
    / SETTINGS['output_subdir']
    / ('debug' if DEBUG_MODE else ''),
)
ARTICLES_PATH = OUTPUT_DIR / SETTINGS['articles_filename']
TITLE_EXTRACTIONS_PATH = OUTPUT_DIR / SETTINGS['title_extractions_filename']
ABSTRACT_EXTRACTIONS_PATH = OUTPUT_DIR / SETTINGS['abstract_extractions_filename']
SUMMARY_PATH = resolve_config_path(PROJECT_ROOT, SETTINGS['summary_path'])

OPENAI_MODEL = configured_env(COMMON, 'openai_model_env', COMMON['openai_model'])
OPENAI_API_KEY = configured_env(COMMON, 'openai_api_key_env')
LLM_MAX_TOKENS = SETTINGS['openai_max_tokens']
LLM_RETRIES = COMMON['llm_retries']
LLM_MAX_CALLS = COMMON['llm_max_calls']
LLM_SAVE_EVERY = COMMON['llm_save_every']
RESUME = COMMON['resume']
RETRY_FAILED_LLM_ROWS = COMMON['retry_failed_llm_rows']
RUN_ABSTRACT_STAGE = SETTINGS['run_abstract_stage']
ABSTRACT_INPUT_MODE = SETTINGS['abstract_input_mode']

pubmed = PubMedClient(
    email=PUBMED_EMAIL,
    api_key=PUBMED_API_KEY,
    tool=COMMON['pubmed_tool'],
    retries=COMMON['pubmed_retries'],
)


## Preview the zoonosis query

In [3]:
print(QUERY)

("zoonosis"[Title/Abstract] OR "zoonotic"[Title/Abstract] OR "zoonoses"[Title/Abstract] OR spillover[Title/Abstract] OR "animal-to-human"[Title/Abstract] OR "animal to human"[Title/Abstract])


## Acquire and persist the PubMed article corpus

The search manifest and article table are written incrementally. A rerun fetches only records that were not completed successfully.

In [4]:
if DEBUG_MODE:
    debug_frames = []
    for debug_index, (debug_pathogen, debug_aliases) in enumerate(DEBUG_PATHOGENS.items()):
        debug_query = PubMedClient.keyword_query(
            [debug_pathogen, *debug_aliases],
            operator='OR',
        )
        debug_articles = collect_zoonosis_articles(
            pubmed,
            OUTPUT_DIR / f'pathogen_{debug_index:02d}',
            query=debug_query,
            max_results=DEBUG_ABSTRACTS_PER_PATHOGEN,
            search_page_size=PUBMED_SEARCH_PAGE_SIZE,
            fetch_batch_size=PUBMED_FETCH_BATCH_SIZE,
            resume=RESUME,
        )
        debug_articles['debug_pathogen'] = debug_pathogen
        debug_frames.append(debug_articles)
    articles = pd.concat(debug_frames, ignore_index=True) if debug_frames else pd.DataFrame()
    articles = limit_debug_abstracts(articles, COMMON)
    ARTICLES_PATH.parent.mkdir(parents=True, exist_ok=True)
    articles.to_parquet(ARTICLES_PATH, index=False)
else:
    articles = collect_zoonosis_articles(
        pubmed,
        OUTPUT_DIR,
        query=QUERY,
        max_results=PUBMED_MAX_RESULTS,
        search_page_size=PUBMED_SEARCH_PAGE_SIZE,
        fetch_batch_size=PUBMED_FETCH_BATCH_SIZE,
        resume=RESUME,
    )
print('PubMed article rows:', len(articles))
display(articles[['pmid', 'title', 'publication_date', 'fetch_status']].head(10))

Fetched 5/5 PubMed records
Fetched 5/5 PubMed records
PubMed article rows: 10


,pmid,title,publication_date,fetch_status
0,12762362,Coxiella burnetii pneumonia.,2003Apr,ok
1,16102309,Coxiella burnetii genotyping.,2005Aug,ok
2,18755387,Q fever.,2008Sep,ok
3,26730641,[Not Available].,2016Feb,ok
4,38133298,Coxiella burnetii Infection in Cats.,2023Dec02,ok
5,18467096,Ecology and genomics of Bacillus subtilis.,2008Jun,ok
6,31976860,A bacterial Goldilocks mechanism.,2020Jan24,ok
7,33098223,Positioning Bacillus subtilis as terpenoid cel...,2021Jun,ok
8,34587446,The Bacillus subtilis Minimal Genome Compendium.,2021Oct15,ok
9,35744626,"Bacillus subtilis Cell Differentiation, Biofil...",2022May27,ok


## Level 1: extract bacterial species from titles

Only titles are supplied to the LLM in this stage. The decision table is checkpointed after every `LLM_SAVE_EVERY` calls and also flushed when the cell is interrupted.

In [5]:
if not OPENAI_API_KEY:
    raise RuntimeError('Set OPENAI_API_KEY before running the LLM extraction stages.')
llm = OpenAIChatCompleter(OPENAI_MODEL)
title_inputs = articles[articles['title'].fillna('').str.strip().ne('')].copy()
title_extractions = run_species_extraction_stage(
    title_inputs,
    llm,
    TITLE_EXTRACTIONS_PATH,
    stage='title',
    model=OPENAI_MODEL,
    max_tokens=LLM_MAX_TOKENS,
    retries=LLM_RETRIES,
    max_llm_calls=LLM_MAX_CALLS,
    save_every=LLM_SAVE_EVERY,
    resume=RESUME,
    retry_failed=RETRY_FAILED_LLM_ROWS,
)
print('Title decisions:', len(title_extractions))
display(title_extractions[['pmid', 'title', 'associated_with_zoonosis', 'bacterial_species', 'confidence']].head(20))

Title extraction 1/10: PMID 12762362
Title extraction 2/10: PMID 16102309
Title extraction 3/10: PMID 18755387
Title extraction 4/10: PMID 26730641
Title extraction 5/10: PMID 38133298
Title extraction 6/10: PMID 18467096
Title extraction 7/10: PMID 31976860
Title extraction 8/10: PMID 33098223
Title extraction 9/10: PMID 34587446
Title extraction 10/10: PMID 35744626
Title decisions: 10


,pmid,title,associated_with_zoonosis,bacterial_species,confidence
0,12762362,Coxiella burnetii pneumonia.,True,"[""Coxiella burnetii""]",0.9
1,16102309,Coxiella burnetii genotyping.,True,"[""Coxiella burnetii""]",0.9
2,18755387,Q fever.,True,"[""Coxiella burnetii""]",0.9
3,26730641,[Not Available].,False,[],1.0
4,38133298,Coxiella burnetii Infection in Cats.,True,"[""Coxiella burnetii""]",0.9
5,18467096,Ecology and genomics of Bacillus subtilis.,False,[],1.0
6,31976860,A bacterial Goldilocks mechanism.,False,[],1.0
7,33098223,Positioning Bacillus subtilis as terpenoid cel...,False,[],1.0
8,34587446,The Bacillus subtilis Minimal Genome Compendium.,False,[],1.0
9,35744626,"Bacillus subtilis Cell Differentiation, Biofil...",False,[],1.0


## Select records for the abstract stage

The default is a precision-first funnel: abstract review is run on articles whose title pass found at least one bacterial species. Set `ABSTRACT_INPUT_MODE = 'all_articles'` above when recall is more important than LLM cost.

In [6]:
def _has_species(value):
    try:
        return bool(json.loads(value))
    except (TypeError, ValueError, json.JSONDecodeError):
        return False

if ABSTRACT_INPUT_MODE == 'all_articles':
    abstract_pmids = set(articles['pmid'].astype(str))
elif ABSTRACT_INPUT_MODE == 'title_positive':
    abstract_pmids = set(
        title_extractions.loc[
            title_extractions['associated_with_zoonosis'].fillna(False)
            & title_extractions['bacterial_species'].map(_has_species),
            'pmid',
        ].astype(str)
    )
else:
    raise ValueError("ABSTRACT_INPUT_MODE must be 'title_positive' or 'all_articles'.")

abstract_inputs = articles[
    articles['pmid'].astype(str).isin(abstract_pmids)
    & articles['abstract'].fillna('').str.strip().ne('')
].copy()
print('Abstract-stage input rows:', len(abstract_inputs))
display(abstract_inputs[['pmid', 'title']].head(20))

Abstract-stage input rows: 4


,pmid,title
0,12762362,Coxiella burnetii pneumonia.
1,16102309,Coxiella burnetii genotyping.
2,18755387,Q fever.
4,38133298,Coxiella burnetii Infection in Cats.


## Level 2: extract bacterial species from abstracts

In [7]:
if RUN_ABSTRACT_STAGE:
    abstract_extractions = run_species_extraction_stage(
        abstract_inputs,
        llm,
        ABSTRACT_EXTRACTIONS_PATH,
        stage='abstract',
        model=OPENAI_MODEL,
        max_tokens=LLM_MAX_TOKENS,
        retries=LLM_RETRIES,
        max_llm_calls=LLM_MAX_CALLS,
        save_every=LLM_SAVE_EVERY,
        resume=RESUME,
        retry_failed=RETRY_FAILED_LLM_ROWS,
    )
    print('Abstract decisions:', len(abstract_extractions))
    display(abstract_extractions[['pmid', 'title', 'associated_with_zoonosis', 'bacterial_species', 'confidence']].head(20))
else:
    print('Abstract stage disabled; the recovery cell can still build a title-only summary.')

Abstract extraction 1/4: PMID 12762362
Abstract extraction 2/4: PMID 16102309
Abstract extraction 3/4: PMID 18755387
Abstract extraction 4/4: PMID 38133298
Abstract decisions: 4


,pmid,title,associated_with_zoonosis,bacterial_species,confidence
0,12762362,Coxiella burnetii pneumonia.,True,"[""Coxiella burnetii""]",1.0
1,16102309,Coxiella burnetii genotyping.,True,"[""Coxiella burnetii""]",0.9
2,18755387,Q fever.,True,"[""Coxiella burnetii""]",1.0
3,38133298,Coxiella burnetii Infection in Cats.,True,"[""Coxiella burnetii""]",1.0


## Persist the species dataframe

In [8]:
species_summary = build_bacterial_species_summary(
    title_extractions,
    abstract_extractions if RUN_ABSTRACT_STAGE else pd.DataFrame(),
    SUMMARY_PATH,
)
print('Bacterial species:', len(species_summary))
display(species_summary)

Bacterial species: 1


,bacterial_species,evidence_levels,title_evidence_count,abstract_evidence_count,evidence_count,pmids,example_titles,updated_at
0,Coxiella burnetii,"[""abstract"", ""title""]",4,4,8,"[""12762362"", ""16102309"", ""18755387"", ""38133298""]","[""Coxiella burnetii Infection in Cats."", ""Coxi...",2026-08-01T12:55:28.573537+00:00


## Rebuild from persisted files after an interrupt

Run this cell independently after stopping a title or abstract extraction cell. It does not use the article or extraction variables from memory.

In [9]:
def _load_checkpoint(path):
    return pd.read_parquet(path) if path.exists() else pd.DataFrame()

persisted_title_extractions = _load_checkpoint(TITLE_EXTRACTIONS_PATH)
persisted_abstract_extractions = _load_checkpoint(ABSTRACT_EXTRACTIONS_PATH)
if persisted_title_extractions.empty and persisted_abstract_extractions.empty:
    raise FileNotFoundError(
        'No extraction checkpoint found. Run the title stage first, then rerun this cell.'
    )
species_summary = build_bacterial_species_summary(
    persisted_title_extractions,
    persisted_abstract_extractions,
    SUMMARY_PATH,
)
print('Rebuilt from checkpoints:')
print('  title rows:', len(persisted_title_extractions))
print('  abstract rows:', len(persisted_abstract_extractions))
print('  species rows:', len(species_summary))
print('  saved to:', SUMMARY_PATH)
display(species_summary)

Rebuilt from checkpoints:
  title rows: 10
  abstract rows: 4
  species rows: 1
  saved to: /Users/f.costa/Code/RecursiveFraming/assets/zoonotic_bacterial_species.parquet


,bacterial_species,evidence_levels,title_evidence_count,abstract_evidence_count,evidence_count,pmids,example_titles,updated_at
0,Coxiella burnetii,"[""abstract"", ""title""]",4,4,8,"[""12762362"", ""16102309"", ""18755387"", ""38133298""]","[""Coxiella burnetii Infection in Cats."", ""Coxi...",2026-08-01T12:55:28.604669+00:00


In [10]:
print('Checkpoint directory:', OUTPUT_DIR)
print('Files:', sorted(path.name for path in OUTPUT_DIR.glob('*') if path.is_file()))
print('Species dataframe exists:', SUMMARY_PATH.exists())

Checkpoint directory: /Users/f.costa/Code/RecursiveFraming/outputs/pubmed_screening/zoonosis_species/debug
Files: ['abstract_species_extraction.parquet', 'title_species_extraction.parquet', 'zoonosis_pubmed_articles.parquet']
Species dataframe exists: True
